# Create submission files

For a simulations with ekbatch you need an init file containing the stimuli and the CVs of the regions from a CARP simulation, you will need:
-  vtx file for a stimulus
-  a set of tags with the conduction velocities

In [46]:
import json
import numpy as np

def json_to_init(stimuli, tag_file, json_param_file, init_file_name):

    # Read tags
    f_input = open(tag_file,"r")
    tags = json.load(f_input)
    f_input.close()

    # Read CVs
    f_input = open(json_param_file,"r")
    params = json.load(f_input)
    f_input.close()

    tags_ventricles_names = ["LV", "RV"]
    CV_ventricle_name = "CV_ventricles"
    k_ventricles_name = "k_ventricles"

    tags_FEC_names = ["FEC_LV", "FEC_RV", "FEC_SV"]
    k_FEC_name = "k_FEC"

    tags_atria_names = ["LA", "RA"]
    CV_atria_name = "CV_atria"
    k_atria_name = "k_atria"

    tags_bachmann_names = ["BB"]
    k_BB_name = "k_BB"

    vtx = []
    nVtx = 0

    for vtxFile in stimuli:
        temp = np.loadtxt(vtxFile, dtype=int, skiprows=2, ndmin=1)
        vtx.append(temp)
        nVtx += temp.shape[0]

    # write .init file
    f = open(init_file_name,'w')

    # header
    f.write('vf:0 vs:0 vn:0 vPS:0\n') # Default properties for tags not specified
    f.write('retro_delay:0 antero_delay:0\n') # If there's no 1D purkinje system, it's ignored.
    # number of stimuli and regions
    f.write('%d %d\n' % (int(nVtx), int(len(tags_ventricles_names)) + len(tags_FEC_names) + len(tags_atria_names) + len(tags_bachmann_names)))
    # stimulus
    for i in range(len(vtx)):
        if len(vtx[i]) == 1:
            f.write('%d %f\n' % (vtx[i],0))
        else:
            for n in vtx[i]:
                f.write('%d %f\n' % (int(n),0))
                
    return_tags_str = ''
    # ek regions
    for i,tag_name in enumerate(tags_ventricles_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_FEC_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_atria_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_bachmann_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    f.close()
    
    return return_tags_str[1:]

In [47]:
import os

heart_folder = "/media/croderog/SeagateExpansionDrive/HCM/10RB00080/"
scenario = 31
Nsim = 180

stimuli = [f'{heart_folder}/sims_folder/fascicles_lv.vtx',
                f'{heart_folder}/sims_folder/fascicles_rv.vtx',
                f'{heart_folder}/sims_folder/SAN.vtx']

json_param_path        = f'{heart_folder}/scenarios/{scenario}/json_files/'
tag_file        = f'{json_param_path}/tags_EP.json'
init_file_path  = f'{heart_folder}/scenarios/{scenario}/data/init_files'

os.system("mkdir -p " + init_file_path)

for sim_num in range(Nsim):
    tags_activated = json_to_init(stimuli=stimuli,
                tag_file=tag_file,
                json_param_file=os.path.join(json_param_path,str(sim_num) + '.json'),
                init_file_name=os.path.join(init_file_path,str(sim_num) + '.init')
                )

# Run simulations

In [48]:


sims_folder = f'{heart_folder}/scenarios/{scenario}/simulations'

meshname = f'{heart_folder}/pre_simulation/myocardium_AV_FEC_BB_lvrv'


cmd = ['ekbatch',meshname]
init_cmd = ','.join([os.path.join(init_file_path,str(sim_num)) for sim_num in range(Nsim)])

os.system(' '.join(cmd+[init_cmd] + [tags_activated]))

os.makedirs(sims_folder,exist_ok=True)
for sim_num in range(Nsim):
    os.system('mv ' + os.path.join(init_file_path,str(sim_num) + '.dat ') + sims_folder)


Executable ID: ICL_LHR_CARPENTRY
Found license file path: /home/croderog/software/CARPentry_ICL_latest/license/license.bin
Using OpenMP parallelization with 24 threads.
Reading mesh ..
Reading elements (txt):                           [==============================]
Reading points (txt):                             [==============================]
Reading fibers (txt):                             [==============================]
Needed 16.3386 seconds for mesh-reading and subdomain-extraction

The simulation domain consists of:
2554123	elements
488779	nodes

Parsed init file: /media/croderog/SeagateExpansionDrive/HCM/10RB00080//scenarios/31/data/init_files/0.init
The used velocities (in m/s) are:
Fiber direction:	0
Sheet direction:	0
Normal direction:	0
Purkinje system:	0
The used junction delays (in ms) are:
Anterograde delay:	0
Retrograde delay:	0

Solving ..
Eikonal solve progress:                           [==============================]
Needed 2.35829 seconds
Wrote /media/croder

# Extract the output

In [49]:
# Extracted from Marina's library

def electrophysiology_output(basefolder,
							 elem_file,
							 tags,
	   						 start_sample=0,
	   						 last_sample=1,
	   						 output_file='Y.txt'):

	print('Reading mesh elem file...')
	elem = np.loadtxt(elem_file,dtype=int,usecols=[1,2,3,4,5],skiprows=1)
	print('Done.')

	V_EIDX = np.where(np.isin(elem[:,-1],tags["ventricles"]+tags["fast_endo"])==1)[0]
	A_EIDX = np.where(np.isin(elem[:,-1],tags["atria"]+tags["bachmann_bundle"])==1)[0]

	V_VTX = np.unique(elem[V_EIDX,0:4].flatten())
	A_VTX = np.unique(elem[A_EIDX,0:4].flatten())

	output = np.zeros((last_sample-start_sample+1,2))

	count = 0
	for i in range(start_sample,last_sample+1):
		print('Computing output for '+str(i)+'.dat...')
		AT=np.loadtxt(os.path.join(basefolder,str(i)+".dat"),dtype=float)
		if (np.min(AT[V_VTX]<0)):
			raise Exception("The ventricles contain a negative activation time.")
		if (np.min(AT[A_VTX]<0)):
			raise Exception("The atria contain a negative activation time.")
			
		output[count,0] = np.max(AT[A_VTX])-np.min(AT[A_VTX])
        
		output[count,1] = np.max(AT[V_VTX])-np.min(AT[V_VTX])
		count += 1

	np.savetxt(output_file,output,fmt="%g")

In [50]:
import json
import numpy as np
import os

basefolder = sims_folder
elem_file = f"{meshname}.elem"

f_input = open(tag_file,"r")
tags = json.load(f_input)
f_input.close()


tags_modified = tags.copy()
tags_modified["ventricles"] = [tags_modified["LV"], tags_modified["RV"]]
tags_modified["fast_endo"] = [tags_modified["FEC_RV"], tags_modified["FEC_SV"]]
tags_modified["atria"] = [tags_modified["LA"], tags_modified["RA"]]
tags_modified["bachmann_bundle"] = [tags_modified["BB"]]

output_path = f'{heart_folder}/scenarios/{scenario}/output'

os.makedirs(output_path,exist_ok=True)

electrophysiology_output(basefolder=basefolder,
							elem_file=elem_file,
							tags=tags_modified,
							start_sample=0,
							last_sample=Nsim-1,
							output_file=os.path.join(output_path,'Y.txt'))

Reading mesh elem file...
Done.
Computing output for 0.dat...
Computing output for 1.dat...
Computing output for 2.dat...
Computing output for 3.dat...
Computing output for 4.dat...
Computing output for 5.dat...
Computing output for 6.dat...
Computing output for 7.dat...
Computing output for 8.dat...
Computing output for 9.dat...
Computing output for 10.dat...
Computing output for 11.dat...
Computing output for 12.dat...
Computing output for 13.dat...
Computing output for 14.dat...
Computing output for 15.dat...
Computing output for 16.dat...
Computing output for 17.dat...
Computing output for 18.dat...
Computing output for 19.dat...
Computing output for 20.dat...
Computing output for 21.dat...
Computing output for 22.dat...
Computing output for 23.dat...
Computing output for 24.dat...
Computing output for 25.dat...
Computing output for 26.dat...
Computing output for 27.dat...
Computing output for 28.dat...
Computing output for 29.dat...
Computing output for 30.dat...
Computing output 

# Make animation of the EP simulation

In [2]:
import json
import math
import numpy as np
import pyvista as pv
import tqdm
import vtk

def read_elem(filename,el_type='Tt',tags=True):
	print('Reading '+filename+'...')

	if el_type=='Tt':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4,5))
		else:
			filtered_lines = []
			with open(filename, 'r') as infile:
				first_line = True
				for line in infile:
					if first_line:
						first_line = False
						continue
					else:
					# Split the line into columns
						columns = line.split()
						# Check if the number of columns is 6
						if len(columns) == 6:
							filtered_lines.append(columns[1:5])
						else:
							break
    
			# Convert the filtered lines to a numpy array
			# Skipping the first row (header) and using specific columns
			data = np.array(filtered_lines, dtype=int)
			return data
			# return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4))
	elif el_type=='Tr':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3,4))
		else:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3))
	elif el_type=='Ln':
		if tags:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2,3))
		else:
			return np.loadtxt(filename, dtype=int, skiprows=1, usecols=(1,2))
	else:
		raise Exception('element type not recognised. Accepted: Tt, Tr, Ln')

def carp_to_pyvista(meshname):

	pts = np.loadtxt(meshname+'.pts', dtype=float, skiprows=1)
	elem = read_elem(meshname+'.elem',el_type='Tt',tags=False)

	tets = np.column_stack((np.ones((elem.shape[0],),dtype=int)*4,elem)).flatten()
	cell_type = np.ones((elem.shape[0],),dtype=int)*vtk.VTK_TETRA	

	plt_msh = pv.UnstructuredGrid(tets,cell_type,pts)

	return plt_msh

def numpy_hook(dct):
	for key, value in dct.items():
		if isinstance(value, list):
			value = np.array(value)
			dct[key] = value
	return dct

def load_json(filename):
	print('Reading '+filename+'...')

	dct = {}
	with open(filename, "r") as f:
		dct = json.load(f, object_hook=numpy_hook)
	return dct

def print_screenshot_video(plt_msh,
						   binary_vector,
						   screenshot_name,
						   camera_settings,
						   title=None,
						   fig_w=1200,
						   fig_h=1200,
						   inactive_color="gray",
						   active_color="darkred",
						   view="anterior",
						   opacity=1.0):

	plotter = pv.Plotter(off_screen=True)
	plotter.background_color = 'white'

	plt_msh.point_data["at"] = binary_vector

	msh = plotter.add_mesh(plt_msh,opacity=opacity,
						   scalars="at",
						   cmap=[inactive_color,active_color],
						   clim=np.array([0.,1.]))

	plotter.remove_scalar_bar()

	plotter.camera.azimuth = camera_settings[view]["azimuth"]
	plotter.camera.elevation = camera_settings[view]["elevation"]

	plotter.add_title(title,
					  font_size=12,
					  font="arial",
					  color="black")
	print("Printing...")
	plotter.screenshot(filename=screenshot_name, 
					   transparent_background=None, 
					   return_img=True,
					   window_size=[fig_w,fig_h])
	print("Printed")
	plotter.close()

def make_activation_video(meshname,
						  activation_file,
						  video_folder,
						  camera_file,
					 	  inactive_color="lightgray",
					 	  active_color="firebrick",
					 	  view="anterior",
						  opacity=1.0):
	
	camera_settings = load_json(camera_file)

	plt_msh = carp_to_pyvista(meshname)
	
	act = np.loadtxt(activation_file,dtype=float)
	
	t0 = 0 
	tend = math.ceil(np.max(act))

	act[act < 0] = tend+10


	count = 0
	for t in tqdm.tqdm(range(t0,tend+1)):

		binary_vector = (act<=t)

		print_screenshot_video(plt_msh,
					           binary_vector,
					           video_folder+"/act_{:03d}.png".format(count),
					           camera_settings,
					           title="time = "+str(t)+" ms",
					           fig_w=1200,
					           fig_h=1200,
					           inactive_color=inactive_color,
					           active_color=active_color,
					           view=view,
							   opacity=opacity)

		count += 1

In [4]:
case="h06"
folder="pre_simulation"

for case in ["h03","h04","h13"]:

	make_activation_video(meshname = f"/media/croderog/SeagateExpansionDrive/rodero_healthy/{case}/{folder}/myocardium_AV_FEC_BB_lvrv_pkj",
						activation_file = f"/media/croderog/SeagateExpansionDrive/rodero_healthy/{case}/{folder}/SR_simulation/SR.dat",
						video_folder=f"/media/croderog/SeagateExpansionDrive/rodero_healthy/{case}/{folder}/SR_simulation/",
						camera_file="/media/croderog/SeagateExpansionDrive/rodero_healthy/old_cases/h01_old/cyc_200/video/camera_settings.json",
						inactive_color="whitesmoke",
						active_color="goldenrod" , # dark yellow
						view="anterior",
						opacity=0.8)

Reading /media/croderog/SeagateExpansionDrive/rodero_healthy/old_cases/h01_old/cyc_200/video/camera_settings.json...
Reading /media/croderog/SeagateExpansionDrive/rodero_healthy/h03/pre_simulation/myocardium_AV_FEC_BB_lvrv_pkj.elem...


  0%|          | 0/144 [00:00<?, ?it/s]

Printing...


  1%|          | 1/144 [00:06<14:32,  6.10s/it]

Printed
Printing...


...

Printed
Printing...


 99%|█████████▉| 140/141 [16:06<00:08,  8.29s/it]

Printed
Printing...


100%|██████████| 141/141 [16:13<00:00,  6.90s/it]

Printed
